# Lab 1: Used Car Price Prediction (Regression)

**Task:** predict the resale price of a used car from its age, mileage, fuel type, transmission, engine size and brand.

**Dataset:** UK Used Car Dataset (Kaggle: https://www.kaggle.com/datasets/adityadesai13/used-car-dataset-ford-and-mercedes), included in this folder as `data/used_car_dataset.zip`, exactly as downloaded: one CSV per brand plus a few extra files.

**What we will do:**
1. Load the files, decide which to merge, merge them, and identify the target and the features.
2. Explore: summary statistics, missing values, correlations with price, the price distribution and outliers.
3. Prepare the data: fix problems found in step 2, encode categories, scale numeric features.
4. Fit and compare linear regression, Ridge, Lasso and polynomial regression.
5. Evaluate with $R^2$ on training and test data and look at the residuals.
6. Report the best model and its hyperparameters.

In [ ]:
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)

## 1. Loading the data

The zip file holds several CSVs. Before merging anything, look at what is there: a dataset as downloaded often contains more than the tidy table we want.

In [ ]:
z = zipfile.ZipFile("data/used_car_dataset.zip")
files = z.namelist()
for name in files:
    df = pd.read_csv(z.open(name))
    print(f"{name:20s} {df.shape}  columns: {list(df.columns)}")

Three kinds of file:
- **Nine brand files** (audi, bmw, ford, hyundi, merc, skoda, toyota, vauxhall, vw) with the same columns: model, year, price, transmission, mileage, fuelType, tax, mpg, engineSize. (Hyundai's tax column is called `tax(£)`: we must rename it, or the merge will create two half-empty columns.)
- **`cclass.csv` and `focus.csv`**: single models, without tax and mpg. Are these new cars or repeats of rows in the Mercedes and Ford files?
- **`unclean cclass.csv` and `unclean focus.csv`**: raw scraped versions of those two, with prices like "£30,495".

Let's check whether the C-Class and Focus files add anything:

In [ ]:
read = lambda name: pd.read_csv(z.open(name))
key = ["model", "year", "price", "mileage"]
for small, big in [("cclass.csv", "merc.csv"), ("focus.csv", "ford.csv")]:
    s, b = read(small), read(big)
    found = s.merge(b[key].drop_duplicates(), on=key, how="inner")
    print(f"{small}: {len(found)} of {len(s)} rows also appear in {big}")
print(read("unclean focus.csv").head(3))

Most C-Class and Focus rows are repeats of rows in the brand files, and those files lack two of our columns; the unclean files are scraped versions of the same two models. We merge only the nine brand files, adding a `brand` column taken from the file name.

In [ ]:
brand_files = ["audi.csv", "bmw.csv", "ford.csv", "hyundi.csv", "merc.csv", "skoda.csv", "toyota.csv", "vauxhall.csv", "vw.csv"]
frames = []
for name in brand_files:
    df = read(name).rename(columns={"tax(£)": "tax"})        # make the Hyundai column name match the others
    df["brand"] = name.replace(".csv", "")
    frames.append(df)
cars = pd.concat(frames, ignore_index=True)
print(cars.shape)
cars.head()

**Target:** `price` (in pounds). **Features:** `year` (we will turn it into the car's age), `mileage`, `engineSize`, `tax`, `mpg` (numeric) and `brand`, `model`, `transmission`, `fuelType` (categorical).

## 2. Exploratory data analysis

### Summary statistics

In [ ]:
cars.describe().round(1)

Things to notice: a car from **2060** (an obvious typing error) and one from 1970; engine sizes of **0** (not possible for petrol or diesel cars); mpg values up to several hundred (hybrids and electric cars, measured differently); tax of 0 (exempt cars, plausible). Let's count the problems:

In [ ]:
print("year > 2020:", (cars["year"] > 2020).sum())
print("engineSize == 0:", (cars["engineSize"] == 0).sum(), "| fuel types among them:", cars.loc[cars["engineSize"] == 0, "fuelType"].value_counts().to_dict())
print("duplicate rows:", cars.duplicated().sum())
print("text columns with leading spaces:", (cars["model"].str.startswith(" ")).mean().round(2), "of model values")

### Missing values

In [ ]:
missing = cars.isna().sum()
print(missing[missing > 0] if missing.any() else "no missing values")
plt.figure(figsize=(8, 3))
sns.heatmap(cars.isna().T, cbar=False)
plt.title("missing values (light = missing)")
plt.xticks([])
plt.show()

No missing values, because we renamed Hyundai's tax column before merging. (Try the merge without the rename: the heatmap then shows a block of missing `tax` values for every non-Hyundai car, and a `tax(£)` column missing for everyone else. A "missing value" problem that is really a merging mistake.)

### Correlation of numeric features with price

In [ ]:
numeric = ["year", "mileage", "tax", "mpg", "engineSize"]
print(cars[numeric + ["price"]].corr()["price"].drop("price").sort_values().round(3))
sns.heatmap(cars[numeric + ["price"]].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.show()

Newer cars and bigger engines cost more; mileage and mpg go with lower prices. Note that year and mileage are strongly correlated with each other (older cars have done more miles): correlated features make individual linear-regression weights unstable (multicollinearity), though predictions are fine.

### Price distribution and outliers

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
axes[0].hist(cars["price"], bins=100); axes[0].set_title("price")
axes[1].hist(np.log(cars["price"]), bins=100); axes[1].set_title("log(price)")
axes[2].boxplot(cars["price"], vert=False); axes[2].set_title("price boxplot")
plt.tight_layout(); plt.show()

q1, q3 = cars["price"].quantile([0.25, 0.75])
upper = q3 + 1.5 * (q3 - q1)
print(f"IQR rule: prices above {upper:.0f} are 'outliers': {(cars['price'] > upper).sum()} cars")
print(cars.loc[cars["price"] > upper, "brand"].value_counts().head())

Price is right-skewed: a long tail of expensive cars. The boxplot rule flags a few thousand cars as outliers, but they are mostly genuine expensive Mercedes, BMW and Audi models, not errors, so we keep them. The log of the price is much more symmetric; we come back to that when looking at residuals.

## 3. Preprocessing

### Fixing problems found in EDA
- Remove exact **duplicate rows** (the same advert scraped twice would otherwise appear in both training and test data).
- Remove the car from **2060** (impossible) and cars with **engine size 0** that are not electric (impossible values; there are few, and imputing an engine size would be a guess).
- Strip the leading spaces from text columns.
- Replace `year` by **age** in years. The data was collected in 2020, so age = 2020 - year. (Same information, more natural to read.)

In [ ]:
cars = cars.drop_duplicates()
cars = cars[cars["year"] <= 2020]
cars = cars[(cars["engineSize"] > 0) | (cars["fuelType"] == "Electric")]
for col in ["model", "transmission", "fuelType"]:
    cars[col] = cars[col].str.strip()
cars["age"] = 2020 - cars["year"]
cars = cars.drop(columns="year")
print(cars.shape)

### Train/test split

We split before fitting anything that learns from data (encoders, scalers), so that the test set stays unseen.

In [ ]:
from sklearn.model_selection import train_test_split

X = cars.drop(columns="price")
y = cars["price"]
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
print(x_train.shape, x_test.shape)

### Encoding and scaling

- **Categorical features** get one-hot (dummy) encoding. `model` has almost 200 values, many with very few cars; categories seen fewer than 50 times in the training data are grouped into one "infrequent" column, and categories that appear only in the test data are handled the same way instead of causing an error.
- **Numeric features** are standardised (mean 0, standard deviation 1). Plain linear regression doesn't need it, but Ridge and Lasso penalise the size of the weights, which only makes sense if the features are on the same scale.

Both steps go into a `ColumnTransformer` inside a pipeline, so they are learned from the training data only.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

numeric_cols = ["age", "mileage", "tax", "mpg", "engineSize"]
categorical_cols = ["brand", "model", "transmission", "fuelType"]
preprocess = ColumnTransformer([
    ("num", StandardScaler(), numeric_cols),
    ("cat", OneHotEncoder(min_frequency=50, handle_unknown="infrequent_if_exist"), categorical_cols),
])
print("columns after encoding:", preprocess.fit(x_train).transform(x_train).shape[1])

## 4. Models

### Baseline: linear regression

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import r2_score, mean_absolute_error

def report(name, model):
    p_train, p_test = model.predict(x_train), model.predict(x_test)
    return {"model": name, "train R2": r2_score(y_train, p_train), "test R2": r2_score(y_test, p_test),
            "test MAE": mean_absolute_error(y_test, p_test)}

results = []
linear = Pipeline([("prep", preprocess), ("model", LinearRegression())]).fit(x_train, y_train)
results.append(report("linear regression", linear))
pd.DataFrame(results).round(3)

### Ridge (L2) and Lasso (L1)

Both add a penalty on the size of the weights to the squared error: Ridge on their squares (shrinks all weights), Lasso on their absolute values (can set weights exactly to zero, i.e. drop features). The strength `alpha` is chosen by cross-validation on the training data (3 folds, to keep it quick), over a small grid on a log scale.

In [ ]:
from sklearn.model_selection import GridSearchCV

for name, reg, grid in [("ridge", Ridge(), [0.01, 0.1, 1, 10, 100]),
                        ("lasso", Lasso(max_iter=20000), [0.1, 1, 10, 100])]:
    pipe = Pipeline([("prep", preprocess), ("model", reg)])
    search = GridSearchCV(pipe, {"model__alpha": grid}, cv=3, scoring="r2", n_jobs=-1).fit(x_train, y_train)
    print(name, "CV R2 by alpha:", {a: round(float(r), 4) for a, r in zip(grid, search.cv_results_["mean_test_score"])})
    results.append(report(f"{name} (alpha={search.best_params_['model__alpha']})", search.best_estimator_))
    if name == "lasso":
        lasso = search.best_estimator_
pd.DataFrame(results).round(3)

With about 78,000 training cars and 151 columns, there is little to regularise: cross-validation picks the *smallest* alpha in each grid, i.e. as little penalty as possible, and all three models are almost identical. Penalties matter when there are many features relative to observations. Lasso does drop a few columns:

In [ ]:
coef = lasso[-1].coef_
print(f"Lasso set {np.sum(coef == 0)} of {len(coef)} weights exactly to zero")

### Polynomial regression

A straight line can't bend: in reality price falls fast in the first years of a car's life and slower later. Polynomial features (squares and products of the numeric features) let a linear model fit curves and interactions such as "mileage matters more for older cars". We add them for the numeric features only (with the categories one-hot encoded as before).

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

def poly_pipe(degree, alpha=None):
    prep = ColumnTransformer([
        ("num", Pipeline([("scale", StandardScaler()), ("poly", PolynomialFeatures(degree, include_bias=False))]), numeric_cols),
        ("cat", OneHotEncoder(min_frequency=50, handle_unknown="infrequent_if_exist"), categorical_cols),
    ])
    return Pipeline([("prep", prep), ("model", LinearRegression() if alpha is None else Ridge(alpha=alpha))])

for degree in [2, 3]:
    results.append(report(f"polynomial degree {degree}", poly_pipe(degree).fit(x_train, y_train)))
pd.DataFrame(results).round(3)

Degree 2 and 3 clearly beat the straight line on the test data: the curves are real. With 70,000 training cars, even degree 3 (55 numeric features) doesn't overfit. Overfitting depends on how much data there is. With only 300 training cars:

In [ ]:
small_x, small_y = x_train.sample(300, random_state=1), y_train.loc[x_train.sample(300, random_state=1).index]
rows = []
for degree in [1, 2, 3, 4]:
    m = poly_pipe(degree).fit(small_x, small_y)
    rows.append({"degree": degree, "train R2 (300 cars)": r2_score(small_y, m.predict(small_x)),
                 "test R2": r2_score(y_test, m.predict(x_test))})
pd.DataFrame(rows).round(3)

The signs of overfitting: as the degree goes up, the training $R^2$ keeps rising (0.69 to 0.95) while the test $R^2$ falls, and from degree 3 it becomes hugely negative: the predictions are worse than just predicting the average price. Two things go wrong at once. The high-degree curves fit the noise of 300 cars, and polynomials **extrapolate** wildly: the test set contains cars with more miles or bigger engines than any of the 300 training cars, and a cubic or quartic curve shoots off to absurd prices outside the range it was fitted on. Does a Ridge penalty help?

In [ ]:
for alpha in [10, 1000]:
    m = poly_pipe(4, alpha=alpha).fit(small_x, small_y)
    print(f"degree 4 + Ridge alpha={alpha}: train R2 {r2_score(small_y, m.predict(small_x)):.3f}, test R2 {r2_score(y_test, m.predict(x_test)):.3f}")

The penalty reduces the damage but doesn't fix it: shrinking the weights can't stop a degree-4 polynomial from exploding far outside the training data. With small data, keep the degree low (or use models that don't extrapolate wildly, such as trees). With the full training data the same degree-3 model was the best of all, because the training data covers almost the whole range of the test data.

## 5. Evaluation: residuals

$R^2$ summarises the fit in one number; residual plots show *where* the model goes wrong. We use the degree-3 model trained on all the training data.

In [ ]:
best_poly = poly_pipe(3).fit(x_train, y_train)
pred = best_poly.predict(x_test)
resid = y_test - pred
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(pred, resid, s=2, alpha=0.3); axes[0].axhline(0, color="k")
axes[0].set_xlabel("predicted price"); axes[0].set_ylabel("residual (actual - predicted)")
axes[1].hist(resid, bins=100); axes[1].set_xlabel("residual")
plt.tight_layout(); plt.show()

The residuals fan out: errors of a few hundred pounds on cheap cars and of several thousand on expensive ones, and the model predicts negative prices for a few very cheap cars. A constant error variance is one of the linear model's assumptions. Price differences between cars are naturally proportional (a 10% depreciation is a few hundred pounds on a cheap car and thousands on a luxury one), which suggests modelling **log(price)** and converting back:

In [ ]:
from sklearn.compose import TransformedTargetRegressor

log_model = TransformedTargetRegressor(regressor=poly_pipe(3), func=np.log, inverse_func=np.exp).fit(x_train, y_train)
results.append(report("polynomial degree 3, log(price)", log_model))
pred_log = log_model.predict(x_test)
plt.scatter(pred_log, y_test - pred_log, s=2, alpha=0.3); plt.axhline(0, color="k")
plt.xlabel("predicted price"); plt.ylabel("residual"); plt.title("log-price model"); plt.show()
pd.DataFrame(results).round(3)

The log model has the best test $R^2$ and the lowest MAE, and never predicts a negative price. The residuals still grow with the price, but now in proportion to it.

## 6. Deliverable: the best model

In [ ]:
summary = pd.DataFrame(results).sort_values("test R2", ascending=False).round(3)
summary

**Best model:** polynomial regression of degree 3 on the numeric features (age, mileage, tax, mpg, engine size), with one-hot encoded brand, model, transmission and fuel type (categories with fewer than 50 cars grouped), trained on log(price). No penalty was needed with this much data: Ridge and Lasso chose small penalties and performed like plain linear regression.

**Things to try:**
- Add a Ridge penalty to the log-price polynomial model and tune `alpha` and the degree together with cross-validation.
- Which brand loses value fastest? Fit the model with an interaction between `age` and `brand`.
- Tree models (decision trees, random forests, gradient boosting) handle non-linearity and interactions without hand-made polynomial features; compare them.